# Combined 4-row figure, reduced to C | Rg | S | K, pooled where possible

Reduced-metric variant of the combined LFC / Degroot-lazy / Degroot-metropolis / BC
figure (portrait + landscape), for both `ws` and `mhk`. Writes into `./figs` in
this folder only — does not touch `../figs/combined_ws` or the notebooks that
produced it (`Figures_4panel_ws.ipynb`, `Figures_4panel_mhk.ipynb`).

Differences from those notebooks:
- `main_props` (panel-(b) correlation curves) trimmed from 8 metrics to just
  `CC`, `Rg`, `l_skewness`, `l_kurtosis` — i.e. only C, R_g, S, K. The
  eigenvalue-extreme metrics (`l_1`, `l_239`, and their un-normalized
  counterparts) are dropped from panel (b).
- **Why S and K look like mirror images of each other in panel (b):**
  skewness and kurtosis of a graph's own normalized-Laplacian eigenvalue
  spectrum satisfy the classical inequality Kurt ≥ Skew² + 1 (equivalently,
  excess-kurtosis form α² ≤ β + 2), proved in
  Rohatgi, V. K., & Székely, G. J. (1989). *"Sharp inequalities between
  skewness and kurtosis."* Statistics & Probability Letters, 8(4), 297–299.
  Empirically, every graph in both families sits close to that bound rather
  than merely satisfying it: K ≈ S² + const, with corr(S², K) ≈ 0.99 for WS
  and ≈ 1.00 for MHK (pooled k=16, verified via the persistent kernel). Since
  S is one-signed (always negative) across the entire `p`-sweep for both
  networks, S → S² → K is monotonic over the observed range — S and K can't
  move independently, so corr(S, H²) ≈ −corr(K, H²) at every frequency.
- Rows 1-3 (LFC / Degroot-lazy / Degroot-metropolis) already pool across every
  seed available for a given `k` via `pooled_gains_all`/`pooled_props`
  (100 seeds at k=16) — unchanged from the source notebooks.
- Row 4 (BC) stays single-realization (k=16, seed1): no other seed has a
  `_bc_gains.csv` extracted yet (see project README TODO), so there's nothing
  to pool there.
- The `EXCLUDE_R4_NOISE` toggle that used to mask a noisy BC frequency window
  out of panel (b) is removed entirely — row 4 always plots the full,
  unfiltered correlation curve, for both networks.
- Panel (b) now marks statistical significance: each metric's Spearman
  correlation is computed with `scipy.stats.spearmanr` (rho and p-value
  together, aligned the same way `.corrwith()` aligned them), and a small
  filled dot is overlaid on the line wherever p < 0.05 for that metric at
  that frequency. The line itself is unchanged and always fully opaque.
- Panel (a) now shows a ±1 std shaded band around each collective-response
  curve, computed via the same groupby the mean curve already used (pooled
  across all realizations sharing that (p, freq) - seeds x nodes at k=16 for
  rows 1-3, nodes only for row 4's single BC seed).


In [ ]:
import os
import re
import sys
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# repo root: REPO_ROOT if set, else the nearest ancestor of the cwd holding scripts/LFC
REPO = os.path.abspath(os.environ.get('REPO_ROOT', os.getcwd()))
while not os.path.isdir(os.path.join(REPO, 'scripts', 'LFC')):
    if os.path.dirname(REPO) == REPO:
        raise RuntimeError('run from inside the repository or set REPO_ROOT')
    REPO = os.path.dirname(REPO)
os.chdir(REPO)
sys.path.append(os.path.abspath('scripts/LFC'))  # lfc_data_loader.py lives here
from lfc_data_loader import pooled_props, pooled_gains_all, find_graph_for_p

REALIZATIONS_ROOT = 'nets'
FIG_DIR = 'output/figures'

warnings.filterwarnings("ignore")
plt.rc('text', usetex=True)
plt.rc('text.latex', preamble=r'\usepackage{mathptmx}')

# ── Shared visual config ────────────────────────────────────────────────────
my_new_colors = ['darkslateblue', 'darkcyan', 'coral', 'blue']
main_props    = ['CC', 'Rg', 'l_skewness', 'l_kurtosis']  # C | Rg | S | K only
# S and K trace near-mirror-image correlation curves in panel (b): see the
# intro cell above for why (Kurt >= Skew^2 + 1; Rohatgi & Szekely 1989).
aux_props     = ['rawCC', 'rawSP', 'rawRg', 'k', 'p', 'rawT', 'l_mean', 'l_variance']
props         = main_props + aux_props
prop_label    = {
    'CC':         r'$C$',
    'Rg':         r'$R_g$',
    'l_skewness': r'$S$',
    'l_kurtosis': r'$K$',
}
cf = -3
sf = 0
BC_EPS_VALUES = [0.05, 0.2, 0.3]
SIG_ALPHA = 0.05  # p < SIG_ALPHA marks a frequency as significant on panel (b)

NET_CONFIG = {
    'ws':  dict(specialp=[51, 30, 12, 7],  label='WS'),
    'mhk': dict(specialp=[0, 50, 83, 99], label='MHK'),
}


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Spearman rho + p-value, aligned on shared index (mirrors what .corrwith()
# did implicitly - verified to reproduce identical rho values, confirmed via
# the persistent kernel against row1's flat (ID,p) index and row4's mismatched
# (single-level p vs (ID,freq,p)) index alike, which pandas' own alignment
# resolves by joining on the shared level name).
# ─────────────────────────────────────────────────────────────────────────────
def _corr_and_pvals(props_df, gains_series, cols):
    aligned_props, aligned_gains = props_df.align(gains_series, join='inner', axis=0)
    rhos, pvals = [], []
    for col in cols:
        if len(aligned_gains) < 2:
            rhos.append(np.nan); pvals.append(np.nan)
            continue
        rho, p = spearmanr(aligned_props[col].values, aligned_gains.values)
        rhos.append(rho); pvals.append(p)
    return rhos, pvals


# ─────────────────────────────────────────────────────────────────────────────
# ROW 1  (LFC / all nodes) - pooled across every seed found for k in 2..32
# ─────────────────────────────────────────────────────────────────────────────
def load_row1(net, specialp, in_k=16, model='LFC', root=REALIZATIONS_ROOT, degrees=range(2, 33, 2)):
    ix = pd.IndexSlice
    intended_k = in_k
    nodes = 240

    network_gains = pd.DataFrame(data={'ID': [], 'freq': [], 'p': []}).set_index(['ID', 'freq', 'p'])
    network_props = pd.DataFrame(data={'ID': [], 'p': []}).set_index(['ID', 'p'])

    for k in degrees:
        new_network_gains = pooled_gains_all(model, nodes, net, k, root=root)
        new_props = pooled_props(model, nodes, net, k, root=root)

        new_props['p'] = new_props.index.get_level_values(1)
        new_props['k'] = k
        new_props['rawCC'] = new_props.CC
        new_props['rawSP'] = new_props.SP
        new_props['rawRg'] = new_props.Rg
        new_props['rawT'] = new_props['T']
        new_props.CC = new_props.CC / new_props.CC.max()
        new_props['T'] = new_props['T'] / new_props['T'].max()
        new_props.Rg = new_props.Rg / new_props.Rg.min()
        new_props.SP = new_props.SP / new_props.SP.min()

        for f in new_network_gains.index.unique():
            new_network_gains.loc[f, 'normH2'] = (
                new_network_gains.loc[f].H2 / new_network_gains.loc[f, 'H2'].max()
            ) * (240 / 12)

        if 'k' in new_network_gains.columns:
            new_network_gains.loc[network_gains.isnull().k, 'k'] = k
        else:
            new_network_gains['k'] = k

        network_props = pd.concat([network_props, new_props.loc[:, props]])
        new_network_gains = new_network_gains.reset_index()
        new_network_gains['freq'] = new_network_gains.freq.apply(lambda x: round(x, 5))
        new_network_gains.set_index(['ID', 'freq'], inplace=True)
        network_gains = pd.concat([network_gains, new_network_gains])

    network_props.fillna(value=0, inplace=True)
    network_gains.fillna(value=0, inplace=True)

    freq16 = network_gains.index.get_level_values(1).unique()
    corr_index_16 = pd.MultiIndex.from_tuples(list(zip(
        main_props * len(freq16),
        ['H2'] * len(freq16) * len(main_props),
        sorted(list(freq16) * len(main_props)))))
    network_corr_16 = pd.DataFrame(index=corr_index_16)
    query_string16 = f'k == {intended_k}'
    props_qk = network_props.query(query_string16).loc[:, main_props]
    corr_values_16, pval_values_16 = [], []
    for f in freq16.sort_values():
        rhos, pvals = _corr_and_pvals(props_qk, network_gains.loc[ix[:, f], 'H2'], main_props)
        corr_values_16 += rhos
        pval_values_16 += pvals
    network_corr_16.loc[corr_index_16, 'spearman'] = np.reshape(corr_values_16, (len(corr_values_16), 1))
    network_corr_16.loc[corr_index_16, 'pvalue']   = np.reshape(pval_values_16, (len(pval_values_16), 1))

    network_gains = network_gains.reset_index().set_index(['ID', 'freq', 'p'])
    prop_special = network_props.loc[network_props.k == intended_k].groupby(level=1).mean()
    gain_grouped = network_gains.loc[network_gains.k == intended_k].groupby(level=[2, 1])
    gain_special = gain_grouped.mean()
    gain_special_std = gain_grouped.std()

    return dict(network_corr_16=network_corr_16, specialp=specialp, specialk=intended_k,
                model=model, net=net, prop_special=prop_special, gain_special=gain_special,
                gain_special_std=gain_special_std,
                ylabel_a=r'Collective response $(H^2)$', ylabel_c=r'Correlation $r_s(H^2,\chi)$')


# ─────────────────────────────────────────────────────────────────────────────
# ROWS 2 & 3  (Degroot lazy / metropolis) - pooled across every seed for k in 2..32
# ─────────────────────────────────────────────────────────────────────────────
def load_row2_3(net, specialp, weight='lazy', in_k=16, model='LFC', root=REALIZATIONS_ROOT, degrees=range(2, 33, 2)):
    ix = pd.IndexSlice
    intended_k = in_k
    nodes = 240
    weight_col = {'lazy': 'Lazy', 'metropolis': 'metro'}[weight]

    network_gains = pd.DataFrame(data={'ID': [], 'freq': [], 'p': []}).set_index(['ID', 'freq', 'p'])
    network_props = pd.DataFrame(data={'ID': [], 'p': []}).set_index(['ID', 'p'])

    for k in degrees:
        new_network_gains = pooled_gains_all(model, nodes, net, k, root=root)
        new_network_gains['H2'] = new_network_gains[weight_col]
        new_props = pooled_props(model, nodes, net, k, root=root)

        new_props['p'] = new_props.index.get_level_values(1)
        new_props['k'] = k
        new_props['rawCC'] = new_props.CC
        new_props['rawSP'] = new_props.SP
        new_props['rawRg'] = new_props.Rg
        new_props['rawT'] = new_props['T']
        new_props.CC = new_props.CC / new_props.CC.max()
        new_props['T'] = new_props['T'] / new_props['T'].max()
        new_props.Rg = new_props.Rg / new_props.Rg.min()
        new_props.SP = new_props.SP / new_props.SP.min()

        for f in new_network_gains.index.unique():
            new_network_gains.loc[f, 'normH2'] = (
                new_network_gains.loc[f].H2 / new_network_gains.loc[f, 'H2'].max()
            ) * (240 / 12)

        if 'k' in new_network_gains.columns:
            new_network_gains.loc[network_gains.isnull().k, 'k'] = k
        else:
            new_network_gains['k'] = k

        network_props = pd.concat([network_props, new_props.loc[:, props]])
        new_network_gains = new_network_gains.reset_index()
        new_network_gains['freq'] = new_network_gains.freq.apply(lambda x: round(x, 5))
        new_network_gains.set_index(['ID', 'freq'], inplace=True)
        network_gains = pd.concat([network_gains, new_network_gains])

    network_props.fillna(value=0, inplace=True)
    network_gains.fillna(value=0, inplace=True)

    freq16 = network_gains.index.get_level_values(1).unique()
    corr_index_16 = pd.MultiIndex.from_tuples(list(zip(
        main_props * len(freq16),
        ['H2'] * len(freq16) * len(main_props),
        sorted(list(freq16) * len(main_props)))))
    network_corr_16 = pd.DataFrame(index=corr_index_16)
    query_string16 = f'k == {intended_k}'
    props_qk = network_props.query(query_string16).loc[:, main_props]
    corr_values_16, pval_values_16 = [], []
    for f in freq16.sort_values():
        rhos, pvals = _corr_and_pvals(props_qk, network_gains.loc[ix[:, f], 'H2'], main_props)
        corr_values_16 += rhos
        pval_values_16 += pvals
    network_corr_16.loc[corr_index_16, 'spearman'] = np.reshape(corr_values_16, (len(corr_values_16), 1))
    network_corr_16.loc[corr_index_16, 'pvalue']   = np.reshape(pval_values_16, (len(pval_values_16), 1))

    network_gains = network_gains.reset_index().set_index(['ID', 'freq', 'p'])
    prop_special = network_props.loc[network_props.k == intended_k].groupby(level=1).mean()
    gain_grouped = network_gains.loc[network_gains.k == intended_k].groupby(level=[2, 1])
    gain_special = gain_grouped.mean()
    gain_special_std = gain_grouped.std()

    return dict(network_corr_16=network_corr_16, specialp=specialp, specialk=intended_k,
                model=model, net=net, prop_special=prop_special, gain_special=gain_special,
                gain_special_std=gain_special_std,
                ylabel_a=r'Collective response $(H^2)$', ylabel_c=r'Correlation $r_s(H^2,\chi)$')


# ─────────────────────────────────────────────────────────────────────────────
# ROW 4  (BC) - single realization (k=16, seed1); no frequency-window masking.
# ─────────────────────────────────────────────────────────────────────────────
def load_row4(net, specialp, in_k=16, model='LFC', root=REALIZATIONS_ROOT, eps=0.2):
    ix = pd.IndexSlice
    intended_k = in_k
    nodes = 240
    seed_dir = f'{root}/{model}/{nodes}/{net}/{intended_k}_seed1'

    bc_props = pd.read_csv(f'{seed_dir}_props.csv', sep='\t', index_col=[0, 1])
    network_props = pd.DataFrame(index=bc_props.index)
    network_props['p'] = bc_props.index.get_level_values(1)
    network_props['k'] = intended_k
    network_props['rawCC'] = bc_props['CC']
    network_props['rawSP'] = bc_props['SP']
    network_props['rawRg'] = bc_props['Rg']
    network_props['rawT'] = bc_props['T']
    network_props['l_mean'] = bc_props['l_mean']
    network_props['l_variance'] = bc_props['l_variance']
    network_props['l_skewness'] = bc_props['l_skewness']
    network_props['l_kurtosis'] = bc_props['l_kurtosis']
    network_props['CC'] = network_props['rawCC'] / network_props['rawCC'].max()
    network_props['T'] = network_props['rawT'] / network_props['rawT'].max()
    network_props['Rg'] = network_props['rawRg'] / network_props['rawRg'].min()
    network_props['SP'] = network_props['rawSP'] / network_props['rawSP'].min()

    network_gains = pd.read_csv(f'{seed_dir}_bc_gains.csv', sep='\t', index_col=[1])
    network_gains = network_gains.rename(columns={f'BC_{eps}': 'H2'})
    # BC_0.05/0.2/0.3 were swept on disjoint frequency grids appended into one
    # CSV - drop rows where the selected eps has no value, or every other
    # point is missing and no line segment gets drawn. This is a data-shape
    # cleanup, not a frequency-window mask - no window is masked out below.
    network_gains = network_gains.dropna(subset=['H2'])

    for f in network_gains.index.unique():
        network_gains.loc[f, 'normH2'] = (
            network_gains.loc[f].H2 / network_gains.loc[f, 'H2'].max()
        ) * (240 / 12)
    if 'k' not in network_gains.columns:
        network_gains['k'] = intended_k

    network_gains = network_gains.reset_index()
    network_gains['freq'] = network_gains['freq'].apply(lambda x: round(x, 5))
    network_gains.set_index(['ID', 'freq'], inplace=True)

    freq16 = network_gains.index.get_level_values(1).unique()
    corr_index_16 = pd.MultiIndex.from_tuples(list(zip(
        main_props * len(freq16),
        ['H2'] * len(freq16) * len(main_props),
        sorted(list(freq16) * len(main_props)))))
    network_corr_16 = pd.DataFrame(index=corr_index_16)
    corr_values_16, pval_values_16 = [], []

    prop_special_grouped = network_props.groupby(level=1)[main_props].mean()
    network_gains_indexed = network_gains.reset_index().set_index(['ID', 'freq', 'p'])

    for f in freq16.sort_values():
        try:
            rhos, pvals = _corr_and_pvals(prop_special_grouped, network_gains_indexed.loc[ix[:, f], 'H2'], main_props)
        except Exception:
            rhos, pvals = [np.nan] * len(main_props), [np.nan] * len(main_props)
        corr_values_16 += rhos
        pval_values_16 += pvals

    if len(corr_values_16) > 0:
        network_corr_16.loc[corr_index_16, 'spearman'] = np.reshape(corr_values_16, (len(corr_values_16), 1))
        network_corr_16.loc[corr_index_16, 'pvalue']   = np.reshape(pval_values_16, (len(pval_values_16), 1))

    prop_special = network_props.groupby(level=1).mean()
    gain_grouped = network_gains_indexed.groupby(level=[2, 1])
    gain_special = gain_grouped.mean()
    gain_special_std = gain_grouped.std()

    return dict(network_corr_16=network_corr_16, specialp=specialp, specialk=intended_k,
                model=model, net=net, prop_special=prop_special, gain_special=gain_special,
                gain_special_std=gain_special_std,
                ylabel_a=r'Collective response $(RMS)$', ylabel_c=r'Correlation $r_s(RMS,\chi)$')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Drawing helpers
# ─────────────────────────────────────────────────────────────────────────────
# Simple/Complex arrow+label placement, shared by both panels below. Always
# axes-fraction (transform=ax.transAxes) so panel a (log-scale ylim) and
# panel b (linear ylim) render the annotation at the SAME relative height
# regardless of each panel's own data range - previously each panel used its
# own data-coordinate y-value, which happened to map to different fractions
# (0.0373 vs 0.0784) and so never lined up between columns. The y-values here
# also carry a bigger arrow-to-text gap than the old data-coordinate version
# needed, sized for the current 11pt annotation font.
ARROW_X = (0.0, 0.4425, 0.535, 1.0)
SIMPLE_COMPLEX_Y_ARROW = 0.045
SIMPLE_COMPLEX_Y_TEXT = 0.10


def _draw_simple_complex(ax):
    x0, x1, x2, x3 = ARROW_X
    arrowprops = dict(facecolor='black', shrink=0.01, width=0.005, headwidth=3)
    ax.annotate('', xy=(x0, SIMPLE_COMPLEX_Y_ARROW), xytext=(x1, SIMPLE_COMPLEX_Y_ARROW),
                xycoords='axes fraction', textcoords='axes fraction', arrowprops=arrowprops)
    ax.annotate('', xy=(x3, SIMPLE_COMPLEX_Y_ARROW), xytext=(x2, SIMPLE_COMPLEX_Y_ARROW),
                xycoords='axes fraction', textcoords='axes fraction', arrowprops=arrowprops)
    ax.text((x0 + x1) / 2, SIMPLE_COMPLEX_Y_TEXT, 'Simple', transform=ax.transAxes,
            fontsize=11, ha='center', verticalalignment='center')
    ax.text((x2 + x3) / 2, SIMPLE_COMPLEX_Y_TEXT, 'Complex', transform=ax.transAxes,
            fontsize=11, ha='center', verticalalignment='center')


def draw_subplot_a(ax_a, data, row_num, show_legend=False):
    prop_special = data['prop_special']
    gain_special = data['gain_special']
    gain_special_std = data.get('gain_special_std')
    specialp = data['specialp']

    ax_a.set_xscale('log')
    ax_a.set_yscale('log')
    ax_a.set_xlabel(r'Frequency $(\omega)$', labelpad=2, fontsize=13)
    ax_a.set_ylabel(data['ylabel_a'], labelpad=2.5, fontsize=13)

    for p in gain_special.index.get_level_values(0).unique()[specialp]:
        C_label = str(prop_special.loc[p].rawCC.round(2))
        if len(C_label) < 4:
            C_label += '0'
        r_label = str((prop_special.loc[p].rawRg.mean() / 1000).round(1))
        r_label += r'\! \times \! 10^{3}'
        label_string = (
            fr'${C_label}  |  {r_label}  |'
            fr'    {prop_special.loc[p].l_mean.round(2)}'
            fr'     \,|     {prop_special.loc[p].l_variance.round(2)}'
            fr'    \,|     {prop_special.loc[p].l_skewness.round(2)}'
            fr'    \,|      {prop_special.loc[p].l_kurtosis.round(2)}$'
        )
        y = gain_special.loc[p].H2.iloc[sf:cf]
        line, = ax_a.plot(y, label=label_string)

        # +-1 std shaded band across the pooled realizations (seeds x nodes
        # for rows 1-3 at k=16's 100 seeds; nodes only for row 4's single BC
        # seed). Lower edge clipped above 0 - panel (a) is log-log, so a
        # mean-minus-std that dips <=0 would just vanish from the band rather
        # than erroring, but clipping keeps the visible edge meaningful.
        if gain_special_std is not None:
            y_std = gain_special_std.loc[p].H2.iloc[sf:cf]
            lower = (y - y_std).clip(lower=1e-4)
            upper = y + y_std
            ax_a.fill_between(y.index, lower, upper, color=line.get_color(),
                              alpha=0.2, linewidth=0, zorder=line.get_zorder() - 1)

    if show_legend:
        leg1 = ax_a.legend(
            title=(r'$ \;\,\;\;\; C \;\,\;\,\;\; | \;\,\,\;\,\, R_{g}'
                   r'  \;\:\,\;\,\,\;\,  | \,\;\,\;\ M   \,\;\,\ |'
                   r' \;\,\,\;\,\, V \;\,\,\;\,| \,\;\,\;\ S   \,\;\,\ |'
                   r' \,\;\,\;\ K  $'),
            loc=[0.025, 0.14], ncol=1, borderpad=0.2, markerscale=0.8,
            handlelength=0.9, handletextpad=0.4, labelspacing=0.3, fontsize=10)
        leg1.get_title().set_position((3.55, 0))
        leg1.get_title().set_fontsize('10')
        leg1.get_frame().set_facecolor('white')
        leg1.get_frame().set_alpha(1.0)
        leg1.get_frame().set_edgecolor('white')

    ax_a.set_xlim([0.0001, 2])
    ax_a.set_ylim([0.017, 520])
    ax_a.grid(True, which="major", ls=":")
    ax_a.tick_params(axis='x', labelsize=13)
    ax_a.tick_params(axis='y', labelsize=13)
    ax_a.text(-0.02, 1.03, rf'$\bf{{({row_num}a)}}$', transform=ax_a.transAxes, fontsize=14)
    _draw_simple_complex(ax_a)


def draw_subplot_b(ax_b, data, row_num, show_legend=False):
    network_corr_16 = data['network_corr_16']

    for idx, metric in enumerate(main_props):
        sub = network_corr_16.loc[(metric, 'H2')].iloc[sf:cf]
        color = my_new_colors[idx % 4]
        ax_b.plot(
            sub.index, sub['spearman'],
            label=prop_label[metric],
            zorder=[3, 2, 1, 4][idx % 4],
            c=color,
            linewidth=2,
            markersize=5,
            ls=[':', '--', '-', 'dashdot'][idx % 4],
            markeredgewidth=1,
            markerfacecolor='none')

        # Overlay a small filled dot wherever this metric's Spearman
        # correlation is significant at this frequency (p < SIG_ALPHA) - the
        # line itself is unchanged/always fully opaque.
        sig = sub[sub['pvalue'] < SIG_ALPHA]
        ax_b.plot(
            sig.index, sig['spearman'],
            linestyle='None', marker='o', markersize=3.5,
            markerfacecolor=color, markeredgecolor=color, markeredgewidth=0,
            zorder=10)

    if show_legend:
        leg2 = ax_b.legend(
            title=r'$\chi$', loc=[0.72, 0.4], borderpad=0.2, markerscale=1,
            handlelength=2, handletextpad=0.4, fontsize=12)
        leg2.get_title().set_position((0, 0))
        leg2.get_title().set_fontsize('13')
        leg2.get_frame().set_facecolor('white')
        leg2.get_frame().set_alpha(1.0)
        leg2.get_frame().set_edgecolor('white')

    ax_b.set_xscale('log')
    ax_b.set_ylabel(data['ylabel_c'], labelpad=2.5, fontsize=13)
    ax_b.set_xlabel(r'Frequency $( \omega )$', labelpad=2, fontsize=13)
    ax_b.set_ylim([-1.5, 1.05])
    ax_b.set_xlim([0.0001, 2])
    ax_b.grid(True, which="major", ls=":")
    ax_b.tick_params(axis='x', labelsize=13)
    ax_b.tick_params(axis='y', labelsize=13)
    ax_b.yaxis.set_major_locator(matplotlib.ticker.FixedLocator([-1, 0, 1]))
    ax_b.yaxis.set_minor_locator(matplotlib.ticker.NullLocator())
    ax_b.fill_between(x=[0.015, 2.9], y1=[-4, -4], y2=[-1.72, -1.72], facecolor='w', zorder=-10)
    ax_b.text(-0.02, 1.04, rf'$\bf{{({row_num}b)}}$', transform=ax_b.transAxes, fontsize=14)
    _draw_simple_complex(ax_b)


def draw_c(ax_c, data):
    gain_special = data['gain_special']
    specialp = data['specialp']
    model = data['model']
    net = data['net']
    specialk = data['specialk']

    for p in gain_special.index.get_level_values(0).unique()[specialp]:
        G = find_graph_for_p(model, 240, net, specialk, p, root=REALIZATIONS_ROOT)
        ax_c.hist(G.vp.eig_laplacian_norm.get_array(), bins=100, density=True, alpha=0.5)

    ax_c.set_ylabel(r'Density', labelpad=6, fontsize=13)
    ax_c.set_xlabel(r'Normalized Eigenvalues', labelpad=2, math_fontfamily='cm', fontsize=13)
    # No hardcoded xlim (unlike the old [-0.02, 2.05], sized for the
    # theoretical max of 2 for normalized-Laplacian eigenvalues) - the actual
    # data here never reaches past ~1.4-1.5, so that fixed range left a large
    # blank strip on the right that draw_c_unnorm (which always autoscaled)
    # didn't have. Autoscaling matches draw_c_unnorm's behavior.
    ax_c.grid(False)
    ax_c.tick_params(axis='x', labelsize=13)
    ax_c.tick_params(axis='y', labelsize=13)
    ax_c.text(-0.02, 1.04, r'$\bf{(c)}$', transform=ax_c.transAxes, fontsize=14)


def draw_c_unnorm(ax_c, data):
    gain_special = data['gain_special']
    specialp = data['specialp']
    model = data['model']
    net = data['net']
    specialk = data['specialk']

    for p in gain_special.index.get_level_values(0).unique()[specialp]:
        G = find_graph_for_p(model, 240, net, specialk, p, root=REALIZATIONS_ROOT)
        ax_c.hist(G.vp.eig_laplacian.get_array(), bins=100, density=True, alpha=0.5)

    # Wider tick labels here than in draw_c (e.g. "0.175" vs "3.5"), so the
    # same labelpad that clears draw_c's ticks isn't enough here - the ylabel
    # sat on top of the tick numbers at labelpad=2.5. labelpad=14 overshot the
    # other way, drifting the label past the ticks into the inter-panel gap.
    ax_c.set_ylabel(r'Density', labelpad=8, fontsize=13)
    ax_c.set_xlabel(r'Un-normalized Eigenvalues', labelpad=2, math_fontfamily='cm', fontsize=13)
    ax_c.grid(False)
    ax_c.tick_params(axis='x', labelsize=13)
    ax_c.tick_params(axis='y', labelsize=13)
    ax_c.text(-0.02, 1.04, r'$\bf{(d)}$', transform=ax_c.transAxes, fontsize=14)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure builders
# ─────────────────────────────────────────────────────────────────────────────
def build_portrait(net, data_row1, data_row2, data_row3, bc_eps, specialp):
    data_row4 = load_row4(net, specialp, eps=bc_eps)

    fig = plt.figure(figsize=(14, 22))
    gs = fig.add_gridspec(10, 6, hspace=0.45, wspace=0.85)

    ax_r1_a = fig.add_subplot(gs[0:2, 0:3]); ax_r1_b = fig.add_subplot(gs[0:2, 3:6])
    draw_subplot_a(ax_r1_a, data_row1, 1, show_legend=True)
    draw_subplot_b(ax_r1_b, data_row1, 1, show_legend=True)

    ax_r2_a = fig.add_subplot(gs[2:4, 0:3]); ax_r2_b = fig.add_subplot(gs[2:4, 3:6])
    draw_subplot_a(ax_r2_a, data_row2, 2); draw_subplot_b(ax_r2_b, data_row2, 2)

    ax_r3_a = fig.add_subplot(gs[4:6, 0:3]); ax_r3_b = fig.add_subplot(gs[4:6, 3:6])
    draw_subplot_a(ax_r3_a, data_row3, 3); draw_subplot_b(ax_r3_b, data_row3, 3)

    ax_r4_a = fig.add_subplot(gs[6:8, 0:3]); ax_r4_b = fig.add_subplot(gs[6:8, 3:6])
    draw_subplot_a(ax_r4_a, data_row4, 4)
    draw_subplot_b(ax_r4_b, data_row4, 4)
    ax_r4_a.set_xlim([1e-3, 1e-1]); ax_r4_b.set_xlim([1e-3, 1e-1])

    ax_r5_c = fig.add_subplot(gs[8:10, 0:3])
    draw_c(ax_r5_c, data_row2)
    ax_r5_d = fig.add_subplot(gs[8:10, 3:6])
    draw_c_unnorm(ax_r5_d, data_row2)

    label = NET_CONFIG[net]['label']
    os.makedirs(FIG_DIR, exist_ok=True)
    fig.savefig(f'{FIG_DIR}/{label}_16_240_lfc_lazy_metro_bc_portrait_CRgSK_eps{bc_eps}.pdf', bbox_inches='tight')
    plt.show()


def build_landscape(net, data_row1, data_row2, data_row3, bc_eps, specialp):
    data_row4 = load_row4(net, specialp, eps=bc_eps)
    data_list = [data_row1, data_row2, data_row3, data_row4]

    fig = plt.figure(figsize=(18, 10))
    gs = fig.add_gridspec(3, 12, hspace=0.30, wspace=2.8)

    for i, data in enumerate(data_list):
        ax_a = fig.add_subplot(gs[0, i * 3:(i + 1) * 3])
        draw_subplot_a(ax_a, data, i + 1)
    for i, data in enumerate(data_list):
        ax_b = fig.add_subplot(gs[1, i * 3:(i + 1) * 3])
        draw_subplot_b(ax_b, data, i + 1)

    ax_c = fig.add_subplot(gs[2, 0:6])
    draw_c(ax_c, data_row2)
    ax_d = fig.add_subplot(gs[2, 6:12])
    draw_c_unnorm(ax_d, data_row2)

    label = 'ws' if net == 'ws' else 'mhk'
    os.makedirs(FIG_DIR, exist_ok=True)
    fig.savefig(f'{FIG_DIR}/{label}_combined_landscape_4a_4b_c_CRgSK_eps{bc_eps}.pdf', bbox_inches='tight', dpi=300)
    plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Run: ws  (rows 1-3 pooled across all seeds for k=16; loaded once, reused
# across the 3 BC epsilons since only row 4 is eps-dependent)
# ─────────────────────────────────────────────────────────────────────────────
net = 'ws'
specialp = NET_CONFIG[net]['specialp']
data_row1 = load_row1(net, specialp, in_k=16)
data_row2 = load_row2_3(net, specialp, weight='lazy', in_k=16)
data_row3 = load_row2_3(net, specialp, weight='metropolis', in_k=16)

for bc_eps in BC_EPS_VALUES:
    build_portrait(net, data_row1, data_row2, data_row3, bc_eps, specialp)
    build_landscape(net, data_row1, data_row2, data_row3, bc_eps, specialp)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Run: mhk
# ─────────────────────────────────────────────────────────────────────────────
net = 'mhk'
specialp = NET_CONFIG[net]['specialp']
data_row1 = load_row1(net, specialp, in_k=16)
data_row2 = load_row2_3(net, specialp, weight='lazy', in_k=16)
data_row3 = load_row2_3(net, specialp, weight='metropolis', in_k=16)

for bc_eps in BC_EPS_VALUES:
    build_portrait(net, data_row1, data_row2, data_row3, bc_eps, specialp)
    build_landscape(net, data_row1, data_row2, data_row3, bc_eps, specialp)


## WS figures with the BC frequency filter

Re-runs the `ws` portrait + landscape figures with row 4's noisy BC
frequency window masked out of panel (b), matching
`Figures_4panel_ws.ipynb`'s `EXCLUDE_R4_NOISE = True` behavior. Writes
alongside the unfiltered `ws` figures above, with an `_freqfilt` filename
suffix so neither run overwrites the other.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# WS, with the BC frequency filter applied to row 4 — row 4 rests on a single
# noisy BC realization (k=16, seed1); freqs 0.00287-0.00579 sit in the steep
# gain roll-off where the four p-curves' H2 values are nearly indistinguishable,
# so their rank order (and thus the Spearman correlation in panel (b)) flips
# erratically there. This masks that window out of panel (b) only - panel (a)'s
# gain curves are left untouched, since the H2 values themselves aren't the
# noisy quantity, only the rank correlation derived from them. Window and
# panel-(b)-only scope match Figures_4panel_ws.ipynb's EXCLUDE_R4_NOISE=True.
# ─────────────────────────────────────────────────────────────────────────────
EXCLUDE_LO_R4, EXCLUDE_HI_R4 = 0.00287, 0.00579


def _mask_row4_noise(data_row4):
    nc4 = data_row4['network_corr_16']
    freqs = nc4.index.get_level_values(2)
    mask = (freqs >= EXCLUDE_LO_R4) & (freqs <= EXCLUDE_HI_R4)
    data_row4['network_corr_16'] = nc4[~mask]
    return data_row4


def build_portrait_freqfilt(net, data_row1, data_row2, data_row3, bc_eps, specialp):
    data_row4 = _mask_row4_noise(load_row4(net, specialp, eps=bc_eps))

    fig = plt.figure(figsize=(14, 22))
    gs = fig.add_gridspec(10, 6, hspace=0.45, wspace=0.85)

    ax_r1_a = fig.add_subplot(gs[0:2, 0:3]); ax_r1_b = fig.add_subplot(gs[0:2, 3:6])
    draw_subplot_a(ax_r1_a, data_row1, 1, show_legend=True)
    draw_subplot_b(ax_r1_b, data_row1, 1, show_legend=True)

    ax_r2_a = fig.add_subplot(gs[2:4, 0:3]); ax_r2_b = fig.add_subplot(gs[2:4, 3:6])
    draw_subplot_a(ax_r2_a, data_row2, 2); draw_subplot_b(ax_r2_b, data_row2, 2)

    ax_r3_a = fig.add_subplot(gs[4:6, 0:3]); ax_r3_b = fig.add_subplot(gs[4:6, 3:6])
    draw_subplot_a(ax_r3_a, data_row3, 3); draw_subplot_b(ax_r3_b, data_row3, 3)

    ax_r4_a = fig.add_subplot(gs[6:8, 0:3]); ax_r4_b = fig.add_subplot(gs[6:8, 3:6])
    draw_subplot_a(ax_r4_a, data_row4, 4)
    draw_subplot_b(ax_r4_b, data_row4, 4)
    ax_r4_a.set_xlim([1e-3, 1e-1]); ax_r4_b.set_xlim([1e-3, 1e-1])

    ax_r5_c = fig.add_subplot(gs[8:10, 0:3])
    draw_c(ax_r5_c, data_row2)
    ax_r5_d = fig.add_subplot(gs[8:10, 3:6])
    draw_c_unnorm(ax_r5_d, data_row2)

    label = NET_CONFIG[net]['label']
    os.makedirs(FIG_DIR, exist_ok=True)
    fig.savefig(f'{FIG_DIR}/{label}_16_240_lfc_lazy_metro_bc_portrait_CRgSK_eps{bc_eps}_freqfilt.pdf', bbox_inches='tight')
    plt.show()


def build_landscape_freqfilt(net, data_row1, data_row2, data_row3, bc_eps, specialp):
    data_row4 = _mask_row4_noise(load_row4(net, specialp, eps=bc_eps))
    data_list = [data_row1, data_row2, data_row3, data_row4]

    fig = plt.figure(figsize=(18, 10))
    gs = fig.add_gridspec(3, 12, hspace=0.30, wspace=2.8)

    for i, data in enumerate(data_list):
        ax_a = fig.add_subplot(gs[0, i * 3:(i + 1) * 3])
        draw_subplot_a(ax_a, data, i + 1)
    for i, data in enumerate(data_list):
        ax_b = fig.add_subplot(gs[1, i * 3:(i + 1) * 3])
        draw_subplot_b(ax_b, data, i + 1)

    ax_c = fig.add_subplot(gs[2, 0:6])
    draw_c(ax_c, data_row2)
    ax_d = fig.add_subplot(gs[2, 6:12])
    draw_c_unnorm(ax_d, data_row2)

    label = 'ws' if net == 'ws' else 'mhk'
    os.makedirs(FIG_DIR, exist_ok=True)
    fig.savefig(f'{FIG_DIR}/{label}_combined_landscape_4a_4b_c_CRgSK_eps{bc_eps}_freqfilt.pdf', bbox_inches='tight', dpi=300)
    plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# Run: ws, with the BC frequency filter applied to row 4 (rows 1-3 pooled
# across all seeds for k=16; loaded fresh here since this cell can run
# independently of the unfiltered ws/mhk runs above)
# ─────────────────────────────────────────────────────────────────────────────
net = 'ws'
specialp = NET_CONFIG[net]['specialp']
data_row1 = load_row1(net, specialp, in_k=16)
data_row2 = load_row2_3(net, specialp, weight='lazy', in_k=16)
data_row3 = load_row2_3(net, specialp, weight='metropolis', in_k=16)

for bc_eps in BC_EPS_VALUES:
    build_portrait_freqfilt(net, data_row1, data_row2, data_row3, bc_eps, specialp)
    build_landscape_freqfilt(net, data_row1, data_row2, data_row3, bc_eps, specialp)
